# 🛒 Online Retail Customer Segmentation
## Capstone Project — K-Means Clustering with RFM Analysis

---

| | |
|---|---|
| **Author** | Krupal Gohil |
| **Dataset** | [UCI Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii) |
| **Algorithm** | K-Means Clustering (Scikit-Learn) |
| **Domain** | E-Commerce / Customer Analytics |
| **Tags** | `Unsupervised Learning` `RFM` `Customer Segmentation` `Data Cleaning` |

---

## 📋 Table of Contents
1. [Project Background & Objectives](#1)
2. [Business Questions](#2)
3. [Dataset Overview](#3)
4. [Imports & Setup](#4)
5. [Data Loading & Exploration](#5)
6. [Data Quality Investigation](#6)
7. [Data Cleaning](#7)
8. [Feature Engineering — RFM Metrics](#8)
9. [Outlier Analysis](#9)
10. [Scaling & Preprocessing](#10)
11. [K-Means Clustering Model](#11)
12. [Cluster Profiling & Labeling](#12)
13. [Outlier Customer Segments](#13)
14. [Final Dashboard — All Segments](#14)
15. [Business Recommendations](#15)
16. [Conclusion & Next Steps](#16)


---
<a id='1'></a>
## 1. Project Background & Objectives

### 🏢 Business Context
A UK-based online gift retailer operates across 40 countries, selling primarily to wholesalers and B2B customers. With over **500,000 transactions** spanning 2009–2010, the company has rich transactional data but lacks a structured understanding of its customer base.

Without knowing *who* the most valuable customers are, the marketing team cannot:
- Allocate retention budgets efficiently
- Personalise communication or offers
- Predict churn before it happens
- Identify high-potential customers worth nurturing

### 🎯 Project Objective
Apply **K-Means Clustering** to segment customers based on **RFM (Recency, Frequency, Monetary)** behaviour — an industry-standard CRM framework — and translate those segments into **actionable business strategies**.

### 🧠 Why K-Means?
| Factor | Justification |
|--------|--------------|
| Unsupervised | No pre-existing labels for customer types |
| Scalable | Works well on 4,000+ customers post-aggregation |
| Interpretable | Cluster centroids are easy to explain to stakeholders |
| RFM-compatible | Euclidean distance is meaningful on normalised RFM space |

### 📦 Deliverables
- Clean, production-ready data pipeline
- 7 named customer segments (4 core K-Means + 3 premium outlier)
- Per-segment marketing strategy recommendations
- A reusable template for ongoing monthly segmentation


---
<a id='2'></a>
## 2. Business Questions

These are the guiding questions this project aims to answer:

### 🔍 Exploratory Questions
> **Q1.** What does the distribution of customer spending look like — is it uniform or heavily skewed toward a small group of big spenders?

> **Q2.** How frequently do customers return to purchase, and what is the typical gap between purchases?

> **Q3.** Which countries contribute the most transactions and revenue?

### 🤖 Clustering Questions
> **Q4.** How many distinct customer segments naturally exist in the data?

> **Q5.** What RFM characteristics define each segment — are some segments high-value but inactive?

> **Q6.** What percentage of customers fall into premium (outlier) segments vs mainstream clusters?

### 💼 Business Strategy Questions
> **Q7.** Which customer segments are at risk of churning and need immediate re-engagement?

> **Q8.** Who are the "Champions" — customers who buy frequently, recently, and spend the most?

> **Q9.** Are there dormant high-value customers who could be reactivated with the right campaign?

> **Q10.** How should the marketing budget be allocated across segments to maximise ROI?


---
<a id='3'></a>
## 3. Dataset Overview

**Source:** UCI Machine Learning Repository — [Online Retail II Dataset](https://archive.ics.uci.edu/dataset/502/online+retail+ii)

### 📊 Column Descriptions

| Column | Type | Description |
|--------|------|-------------|
| `Invoice` | string | Unique 6-digit invoice number. Prefix `C` = cancellation, `A` = bad debt |
| `StockCode` | string | Product code (typically 5 digits, some with suffix letters) |
| `Description` | string | Product name |
| `Quantity` | int | Units per transaction (negative = return/cancellation) |
| `InvoiceDate` | datetime | Date and time of invoice |
| `Price` | float | Unit price in GBP (£) |
| `Customer ID` | float | Unique customer identifier (nullable — guest checkouts have no ID) |
| `Country` | string | Customer country |

### 🗓️ Time Range
- **Start:** December 2009  
- **End:** December 2010  
- **Coverage:** ~13 months of transactional data

### 📏 Raw Dataset Scale
- ~525,000 transaction rows
- ~29,000 unique invoices
- ~4,600 unique stock codes
- Customers from **40 countries**


---
<a id='4'></a>
## 4. Imports & Setup


In [ ]:
# ─── Core Data Libraries ────────────────────────────────────────────────────
import numpy as np          # Numerical operations
import pandas as pd         # DataFrame manipulation

# ─── Visualisation Libraries ─────────────────────────────────────────────────
import matplotlib.pyplot as plt   # Base plotting
import seaborn as sns             # Statistical visualisation

# ─── Machine Learning — Clustering & Preprocessing ───────────────────────────
from sklearn.cluster import KMeans              # K-Means algorithm
from sklearn.metrics import silhouette_score    # Cluster quality metric
from sklearn.preprocessing import StandardScaler # Feature normalisation

# ─── Display Settings ────────────────────────────────────────────────────────
pd.options.display.float_format = '{:20.2f}'.format   # Uniform float display
pd.set_option('display.max_columns', 999)             # Show all columns
sns.set_theme(style='whitegrid', palette='muted')     # Clean plot theme
plt.rcParams['figure.dpi'] = 100                      # Sharper inline plots

print("✅ All libraries loaded successfully.")
print(f"   Pandas  : {pd.__version__}")
print(f"   NumPy   : {np.__version__}")


---
<a id='5'></a>
## 5. Data Loading & Exploration

We load **Sheet 0** (Year 2009–2010) from the Online Retail II Excel file and perform an initial inspection of shape, types, and summary statistics.


In [ ]:
# ─── Load dataset ────────────────────────────────────────────────────────────
# Sheet 0 covers transactions from Dec 2009 to Dec 2010
# Update path to match your local file location
df = pd.read_excel('online_retail_II.xlsx', sheet_name=0)

print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


In [ ]:
# ─── Schema inspection ───────────────────────────────────────────────────────
# Key observations:
#  • Customer ID has ~107,000 nulls  → guest checkouts; must be dropped for RFM
#  • Description has ~2,900 nulls    → minor, not used in RFM
#  • InvoiceDate is datetime         ✅ already parsed correctly
df.info()


In [ ]:
# ─── Numerical summary statistics ────────────────────────────────────────────
# Red flags to investigate:
#  • Quantity min = -9,600     → negative quantities = returns/cancellations
#  • Price    min = -53,594    → bad debt adjustments (Invoice prefix A)
#  • Price    mean=4.69 but std=146 → extreme outliers present
df.describe()


In [ ]:
# ─── Categorical column summaries ────────────────────────────────────────────
# 40 unique countries, 4,632 stock codes, 28,816 invoices
# Top product: 85123A - WHITE HANGING HEART T-LIGHT HOLDER (3,516 transactions)
df.describe(include="O")


---
<a id='6'></a>
## 6. Data Quality Investigation

Before cleaning, we systematically investigate each known data quality issue:
1. **Missing Customer IDs** — guest/unregistered transactions
2. **Negative Quantities** — product returns and cancellations
3. **Non-standard Invoice formats** — cancellations (C prefix), adjustments (A prefix)
4. **Non-standard StockCodes** — internal charges, test entries, gift cards


In [ ]:
# ─── Issue 1: Missing Customer IDs ───────────────────────────────────────────
# Rows with no Customer ID cannot be assigned to a specific customer
# → must be excluded from RFM analysis
print(f"Rows missing Customer ID: {df['Customer ID'].isnull().sum():,} "
      f"({df['Customer ID'].isnull().mean()*100:.1f}% of total)")
df[df['Customer ID'].isnull()].head(10)


In [ ]:
# ─── Issue 2: Negative Quantities ────────────────────────────────────────────
# Negative quantities = product returns — often paired with C-prefix invoices
# If kept, they would lower MonetaryValue and distort Frequency in RFM
print(f"Rows with negative Quantity: {(df['Quantity'] < 0).sum():,}")
df[df['Quantity'] < 0].head(10)


In [ ]:
# ─── Issue 3: Invoice Format Anomalies ───────────────────────────────────────
# Standard invoices are 6 numeric digits; anything else is non-standard
df['Invoice'] = df['Invoice'].astype('str')
df[df['Invoice'].str.match("^\\d{6}$") == False].head(10)


In [ ]:
# What non-numeric prefixes exist in the Invoice column?
# Output: ['', 'C', 'A']
#  ''  = standard (6-digit number with no prefix, treated as numeric match)
#  'C' = cancellation invoice
#  'A' = bad debt adjustment (extreme negative prices)
df['Invoice'].str.replace("[0-9]", "", regex=True).unique()


In [ ]:
# A-prefix invoices represent bad debt write-offs
# These 3 entries alone contain prices of -£53K, -£44K, -£38K
# They would catastrophically distort MonetaryValue if included
df[df['Invoice'].str.startswith('A')]


In [ ]:
# ─── Issue 4: Non-standard StockCodes ────────────────────────────────────────
# Standard pattern: 5 digits, optionally followed by letters (e.g. 85123A)
# The items below are internal codes — not physical products sold to customers

df['StockCode'] = df['StockCode'].astype('str')
non_standard = df[
    (df['StockCode'].str.match('^\\d{5}') == False) &
    (df['StockCode'].str.match('^\\d{5}[a-zA-Z]+$') == False)
]['StockCode'].unique()

print(f"Non-standard StockCodes found: {len(non_standard)}")
print(non_standard)


In [ ]:
# DOT = DOTCOM POSTAGE — pure postage charge, no physical product
# 736 such rows — must be excluded to avoid inflating MonetaryValue
df[df['StockCode'].str.contains('^DOT')].head(5)


### StockCode Exclusion Decision Table

| Code | Description | Action |
|------|-------------|--------|
| `DCGS*` | Internal/gift voucher codes | **Exclude** |
| `D` | Discount rows | **Exclude** |
| `DOT` | Dotcom postage charges | **Exclude** |
| `M` / `m` | Manual transactions | **Exclude** |
| `C2`, `C3` | Carriage transactions | **Exclude** |
| `BANK CHARGES` / `B` | Bank charges | **Exclude** |
| `S` | Samples sent to customer | **Exclude** |
| `TEST*` | Test/QA entries | **Exclude** |
| `gift_*` | Gift card purchases (no customer ID) | **Exclude** |
| `PADS` | Legitimate padding product | **Include** |
| `SP1002`, `AMAZONFEE`, `ADJUST*` | Admin/fees | **Exclude** |


---
<a id='7'></a>
## 7. Data Cleaning

### Cleaning Strategy

| Issue | Decision | Reason |
|-------|----------|--------|
| Missing `Customer ID` | **Drop** | Cannot map to a customer for RFM |
| Negative `Quantity` | **Drop** via Invoice filter | Returns distort revenue and frequency |
| `Invoice` prefix `C` | **Drop** | Cancellation, not a purchase |
| `Invoice` prefix `A` | **Drop** | Bad debt write-off, not a sale |
| Non-product `StockCode` | **Drop** | Internal charges, not customer purchases |
| `PADS` StockCode | **Keep** | Legitimate product code |
| `Price == 0` | **Drop** | Free samples / data errors, not useful for RFM |

> **Expected retention:** ~77% of rows — acceptable for a real-world dataset.


In [ ]:
# ─── Step 1: Work on a copy to preserve raw data ─────────────────────────────
cleaned_df = df.copy()
print(f"Starting rows: {len(cleaned_df):,}")


In [ ]:
# ─── Step 2: Keep only standard 6-digit numeric invoices ─────────────────────
# Removes C-prefix (cancellations) and A-prefix (bad debt adjustments)
cleaned_df["Invoice"] = cleaned_df["Invoice"].astype("str")
mask = cleaned_df["Invoice"].str.match("^\\d{6}$") == True
cleaned_df = cleaned_df[mask]
print(f"After removing non-standard invoices: {len(cleaned_df):,} rows")


In [ ]:
# ─── Step 3: Keep only valid product StockCodes ──────────────────────────────
# Allow: 5 digits, 5 digits + letters, and the special 'PADS' code
cleaned_df["StockCode"] = cleaned_df["StockCode"].astype("str")
mask = (
    (cleaned_df['StockCode'].str.match("^\\d{5}$") == True)
    | (cleaned_df["StockCode"].str.match("^\\d{5}[a-zA-Z]+$") == True)
    | (cleaned_df["StockCode"].str.match("^PADS$") == True)
)
cleaned_df = cleaned_df[mask]
print(f"After removing non-product StockCodes: {len(cleaned_df):,} rows")


In [ ]:
# ─── Step 4: Drop rows with missing Customer ID ──────────────────────────────
# Guest checkouts have no customer identity — cannot be segmented in RFM
cleaned_df.dropna(subset=['Customer ID'], inplace=True)
print(f"After dropping null Customer IDs: {len(cleaned_df):,} rows")


In [ ]:
# ─── Step 5: Remove zero-price transactions ──────────────────────────────────
# Zero-price items are samples or admin entries — inflate Frequency without value
print(f"Zero-price rows: {len(cleaned_df[cleaned_df['Price'] == 0])}")
cleaned_df = cleaned_df[cleaned_df['Price'] > 0.0]
print(f"After removing zero-price rows: {len(cleaned_df):,} rows")


In [ ]:
# ─── Cleaning Summary ────────────────────────────────────────────────────────
retention_pct = len(cleaned_df) / len(df) * 100
print(f"Raw rows      : {len(df):,}")
print(f"Clean rows    : {len(cleaned_df):,}")
print(f"Rows dropped  : {len(df) - len(cleaned_df):,}")
print(f"Retention rate: {retention_pct:.1f}%")
print()
cleaned_df.describe()


---
<a id='8'></a>
## 8. Feature Engineering — RFM Metrics

### What is RFM?

RFM is a proven CRM framework that scores each customer on three dimensions:

| Dimension | Definition | Business Meaning |
|-----------|-----------|-----------------|
| **Recency (R)** | Days since last purchase | Recent buyers are more likely to respond to outreach |
| **Frequency (F)** | Number of unique invoices | Loyal customers buy often |
| **Monetary (M)** | Total revenue generated (£) | High spenders are more valuable to retain |

### Computation Steps
1. Compute `SalesLineTotal = Price × Quantity` per transaction line
2. Aggregate per customer: total revenue, count of unique invoices, last invoice date
3. Compute Recency as days between last purchase and the dataset reference date

> 💡 **Note on Recency direction:** A *higher* Recency value means the customer purchased *less recently* — potentially at higher churn risk.


In [ ]:
# ─── Step 1: Revenue per transaction line ────────────────────────────────────
# SalesLineTotal = unit price × quantity purchased
# This is the monetary contribution of each individual line in an invoice
# Uses .loc to avoid SettingWithCopyWarning
cleaned_df = cleaned_df.copy()
cleaned_df['SalesLineTotal'] = cleaned_df["Price"] * cleaned_df['Quantity']

cleaned_df[['Customer ID', 'Invoice', 'Quantity', 'Price', 'SalesLineTotal']].head(8)


In [ ]:
# ─── Step 2: Aggregate to customer-level RFM features ────────────────────────
#  MonetaryValue   = sum of all SalesLineTotal per customer
#  Frequency       = count of unique invoices (not line items)
#  LastInvoiceDate = most recent purchase date (used to compute Recency)
aggregated_df = cleaned_df.groupby(by="Customer ID", as_index=False).agg(
    MonetaryValue=("SalesLineTotal", "sum"),
    Frequency=("Invoice", "nunique"),
    LastInvoiceDate=("InvoiceDate", "max")
)

print(f"Unique customers in aggregated dataset: {len(aggregated_df):,}")
aggregated_df.head(5)


In [ ]:
# ─── Step 3: Compute Recency ──────────────────────────────────────────────────
# Reference date = last transaction date in the dataset (snapshot date)
# Recency = number of days since last purchase (lower = more recently active)
max_invoice_date = aggregated_df['LastInvoiceDate'].max()
aggregated_df['Recency'] = (max_invoice_date - aggregated_df['LastInvoiceDate']).dt.days

print(f"Reference / snapshot date : {max_invoice_date.date()}")
print(f"Recency range             : {aggregated_df['Recency'].min()} – {aggregated_df['Recency'].max()} days")
aggregated_df.head(5)


In [ ]:
# ─── Distribution plots for RFM features ─────────────────────────────────────
# Expected: right-skewed distributions due to high-value outliers
# Most customers are low-frequency/low-spend; a few are extremely active
# Answering Business Questions Q1 and Q2

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(aggregated_df['MonetaryValue'], bins=40, color='#4C72B0', edgecolor='white')
axes[0].set_title('💰 Monetary Value Distribution', fontweight='bold')
axes[0].set_xlabel('Total Revenue (£)')
axes[0].set_ylabel('Number of Customers')

axes[1].hist(aggregated_df['Frequency'], bins=30, color='#55A868', edgecolor='white')
axes[1].set_title('🔁 Frequency Distribution', fontweight='bold')
axes[1].set_xlabel('Number of Unique Invoices')
axes[1].set_ylabel('Number of Customers')

axes[2].hist(aggregated_df['Recency'], bins=30, color='#C44E52', edgecolor='white')
axes[2].set_title('📅 Recency Distribution', fontweight='bold')
axes[2].set_xlabel('Days Since Last Purchase')
axes[2].set_ylabel('Number of Customers')

plt.suptitle('RFM Feature Distributions (Pre-Outlier Removal)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 RFM Summary Statistics:")
aggregated_df[['MonetaryValue', 'Frequency', 'Recency']].describe().round(2)


In [ ]:
# ─── Boxplots to visualise outlier severity ──────────────────────────────────
# The long upper whiskers in Monetary and Frequency confirm extreme outliers exist
# These VIP customers will be handled as separate premium segments (Section 9)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(data=aggregated_df['MonetaryValue'], ax=axes[0], color='#4C72B0')
axes[0].set_title('💰 Monetary Value Boxplot', fontweight='bold')

sns.boxplot(data=aggregated_df['Frequency'], ax=axes[1], color='#55A868')
axes[1].set_title('🔁 Frequency Boxplot', fontweight='bold')

sns.boxplot(data=aggregated_df['Recency'], ax=axes[2], color='#C44E52')
axes[2].set_title('📅 Recency Boxplot', fontweight='bold')

plt.suptitle('Outlier Visualisation — RFM Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
<a id='9'></a>
## 9. Outlier Analysis

K-Means is **sensitive to outliers** because it minimises the sum of squared distances. A single extreme-value customer can pull an entire centroid away from the true cluster centre.

### Strategy: Separate, Don't Drop
We use the **IQR (Interquartile Range)** method:
- Lower fence = Q1 − 1.5 × IQR  
- Upper fence = Q3 + 1.5 × IQR

Monetary and Frequency outliers are **separated** and treated as premium segments rather than discarded. This preserves crucial information about the business's most valuable customers.

> **Why not just drop outliers?** High-monetary and high-frequency customers are exactly the VIPs the business should be focussing on. Dropping them would lose strategic insight about the most profitable customer group.


In [ ]:
# ─── Identify Monetary outliers via IQR ──────────────────────────────────────
M_Q1 = aggregated_df["MonetaryValue"].quantile(0.25)
M_Q3 = aggregated_df["MonetaryValue"].quantile(0.75)
M_IQR = M_Q3 - M_Q1

monetary_outliers_df = aggregated_df[
    (aggregated_df["MonetaryValue"] > (M_Q3 + 1.5 * M_IQR)) |
    (aggregated_df["MonetaryValue"] < (M_Q1 - 1.5 * M_IQR))
].copy()

print(f"Monetary outliers identified : {len(monetary_outliers_df):,} customers")
print(f"IQR upper fence              : £{M_Q3 + 1.5 * M_IQR:,.2f}")
monetary_outliers_df.describe()


In [ ]:
# ─── Identify Frequency outliers via IQR ─────────────────────────────────────
F_Q1 = aggregated_df['Frequency'].quantile(0.25)
F_Q3 = aggregated_df['Frequency'].quantile(0.75)
F_IQR = F_Q3 - F_Q1

frequency_outliers_df = aggregated_df[
    (aggregated_df['Frequency'] > (F_Q3 + 1.5 * F_IQR)) |
    (aggregated_df['Frequency'] < (F_Q1 - 1.5 * F_IQR))
].copy()

print(f"Frequency outliers identified : {len(frequency_outliers_df):,} customers")
print(f"IQR upper fence               : {F_Q3 + 1.5 * F_IQR:.1f} invoices")
frequency_outliers_df.describe()


In [ ]:
# ─── Create non-outlier DataFrame for core K-Means clustering ────────────────
# Excludes customers that are outliers in Monetary OR Frequency (or both)
non_outliers_df = aggregated_df[
    (~aggregated_df.index.isin(monetary_outliers_df.index)) &
    (~aggregated_df.index.isin(frequency_outliers_df.index))
].copy()

print(f"Total customers    : {len(aggregated_df):,}")
print(f"Core (non-outlier) : {len(non_outliers_df):,}")
print(f"Outliers separated : {len(aggregated_df) - len(non_outliers_df):,}")
non_outliers_df.describe()


In [ ]:
# ─── Post-removal boxplots — much cleaner RFM distributions ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(data=non_outliers_df['MonetaryValue'], ax=axes[0], color='#4C72B0')
axes[0].set_title('💰 Monetary (Outliers Removed)', fontweight='bold')

sns.boxplot(data=non_outliers_df['Frequency'], ax=axes[1], color='#55A868')
axes[1].set_title('🔁 Frequency (Outliers Removed)', fontweight='bold')

sns.boxplot(data=non_outliers_df['Recency'], ax=axes[2], color='#C44E52')
axes[2].set_title('📅 Recency (Outliers Removed)', fontweight='bold')

plt.suptitle('RFM Distributions After Outlier Separation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
<a id='10'></a>
## 10. Scaling & Preprocessing

K-Means uses **Euclidean distance**, so features must be on comparable scales. Without scaling:
- `MonetaryValue` (£0–£5,000) would dominate cluster assignments
- `Recency` (0–365 days) would be underweighted
- `Frequency` (1–50) would be nearly ignored

We apply `StandardScaler` which transforms each feature to **mean = 0, std = 1**.


In [ ]:
# ─── 3D scatter BEFORE scaling ───────────────────────────────────────────────
# Note the vastly different axis scales — Monetary dominates the distance metric
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(projection="3d")
ax.scatter(non_outliers_df["MonetaryValue"], non_outliers_df["Frequency"],
           non_outliers_df["Recency"], alpha=0.4, s=8, color='steelblue')
ax.set_xlabel('Monetary Value (£)', labelpad=10)
ax.set_ylabel('Frequency', labelpad=10)
ax.set_zlabel('Recency (days)', labelpad=10)
ax.set_title('RFM Space — Before Scaling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─── Apply StandardScaler to the 3 RFM features ─────────────────────────────
# fit_transform: computes mean & std on data, then applies Z-score normalisation
# We preserve the original index so cluster labels can be merged back later
scaler = StandardScaler()
scaled_data = scaler.fit_transform(non_outliers_df[["MonetaryValue", "Frequency", "Recency"]])

scaled_data_df = pd.DataFrame(
    scaled_data,
    index=non_outliers_df.index,
    columns=["MonetaryValue", "Frequency", "Recency"]
)

print("Scaled feature statistics (should show mean ≈ 0, std ≈ 1):")
scaled_data_df.describe().round(3)


In [ ]:
# ─── 3D scatter AFTER scaling ────────────────────────────────────────────────
# All 3 axes now on a similar ±3 range — K-Means can cluster fairly across features
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(projection="3d")
ax.scatter(scaled_data_df["MonetaryValue"], scaled_data_df["Frequency"],
           scaled_data_df["Recency"], alpha=0.4, s=8, color='coral')
ax.set_xlabel('Monetary Value (scaled)', labelpad=10)
ax.set_ylabel('Frequency (scaled)', labelpad=10)
ax.set_zlabel('Recency (scaled)', labelpad=10)
ax.set_title('RFM Space — After Scaling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
<a id='11'></a>
## 11. K-Means Clustering Model

### Choosing the Optimal K

We use two complementary methods to select the number of clusters:

| Method | What it measures | How to read it |
|--------|-----------------|----------------|
| **Elbow Method (Inertia)** | Within-cluster sum of squared distances | Look for the "elbow" — where adding more clusters gives diminishing returns |
| **Silhouette Score** | How well-separated clusters are (range: −1 to +1) | Higher = better defined and more distinct clusters |

These two methods together validate the *natural* number of customer groups in the data. Answering **Business Question Q4**.


In [ ]:
# ─── Sweep K from 2 to 12, recording Inertia and Silhouette Score ────────────
# max_iter=1000 ensures convergence even for tricky random initialisations
max_k = 12
inertia = []
silhouette_scores = []
k_values = range(2, max_k + 1)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, max_iter=1000)
    cluster_labels_temp = kmeans.fit_predict(scaled_data_df)
    sil_score = silhouette_score(scaled_data_df, cluster_labels_temp)
    silhouette_scores.append(sil_score)
    inertia.append(kmeans.inertia_)
    print(f"k={k:2d}  |  Inertia: {kmeans.inertia_:9,.1f}  |  Silhouette: {sil_score:.4f}")

print("\n✅ Sweep complete.")


In [ ]:
# ─── Plot Elbow Curve and Silhouette Scores side by side ─────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Elbow curve
ax1.plot(k_values, inertia, marker='o', color='#4C72B0', linewidth=2)
ax1.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='Chosen k=4')
ax1.set_title('📉 Elbow Method — Inertia vs k', fontweight='bold')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_xticks(list(k_values))
ax1.legend()
ax1.grid(True, alpha=0.4)

# Silhouette scores
ax2.plot(k_values, silhouette_scores, marker='o', color='#DD8452', linewidth=2)
ax2.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='Chosen k=4')
ax2.set_title('📈 Silhouette Score vs k', fontweight='bold')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(list(k_values))
ax2.legend()
ax2.grid(True, alpha=0.4)

plt.suptitle('Optimal K Selection — Elbow + Silhouette', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

best_k = k_values[silhouette_scores.index(max(silhouette_scores))]
print(f"Best k by silhouette: k={best_k} (score: {max(silhouette_scores):.4f})")
print(f"Chosen k=4 based on elbow visual and silhouette plateau analysis.")


In [ ]:
# ─── Train final K-Means model with k=4 ──────────────────────────────────────
# init='k-means++' (default) gives smarter starting centroids than random init
# random_state=42 ensures reproducibility across runs
kmeans = KMeans(n_clusters=4, random_state=42, max_iter=1000)
cluster_labels = kmeans.fit_predict(scaled_data_df)

# Assign cluster IDs back to the original (un-scaled) dataframe
non_outliers_df = non_outliers_df.copy()
non_outliers_df['Cluster'] = cluster_labels

print("Cluster size distribution:")
print(non_outliers_df['Cluster'].value_counts().sort_index().to_string())


In [ ]:
# ─── 3D scatter plot coloured by cluster assignment ──────────────────────────
# Each colour represents one customer segment in the RFM space
cluster_colors_main = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c', 3: '#d62728'}
colors = non_outliers_df['Cluster'].map(cluster_colors_main)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(projection='3d')
ax.scatter(non_outliers_df['MonetaryValue'], non_outliers_df['Frequency'],
           non_outliers_df['Recency'], c=colors, marker='o', alpha=0.5, s=12)
ax.set_xlabel('Monetary Value (£)', labelpad=10)
ax.set_ylabel('Frequency', labelpad=10)
ax.set_zlabel('Recency (days)', labelpad=10)
ax.set_title('Customer Segments in RFM Space (k=4)', fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(color=c, label=f'Cluster {k}') for k, c in cluster_colors_main.items()]
ax.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()


---
<a id='12'></a>
## 12. Cluster Profiling & Labeling

We analyse each cluster's mean RFM values to understand the **customer archetype** they represent, then assign a memorable, action-oriented business label.

### Labeling Framework

| Cluster | Recency | Frequency | Monetary | Label | Rationale |
|---------|---------|-----------|----------|-------|-----------|
| 0 | Medium | Medium | Medium | **RETAIN** | Stable mid-tier — keep engaged |
| 1 | High | Low | Low | **RE-ENGAGE** | Lapsed customers — win-back needed |
| 2 | Low | Low | Low | **NURTURE** | New/infrequent — build the habit |
| 3 | Low | Medium | Higher | **REWARD** | Active loyalists — recognise and reinforce |


In [ ]:
# ─── Violin plots for each RFM feature across the 4 clusters ─────────────────
# Violin plots show both distribution shape AND median — more informative than boxplots
# The inner='quartile' option marks Q1, median, Q3 inside each violin
fig, axes = plt.subplots(3, 1, figsize=(13, 16))

for ax, feature, title in zip(
    axes,
    ['MonetaryValue', 'Frequency', 'Recency'],
    ['💰 Monetary Value by Cluster', '🔁 Frequency by Cluster', '📅 Recency by Cluster']
):
    # Per-cluster violin
    sns.violinplot(x=non_outliers_df['Cluster'], y=non_outliers_df[feature],
                   palette=cluster_colors_main, hue=non_outliers_df['Cluster'],
                   ax=ax, inner='quartile', legend=False)
    # Overall distribution overlay in light grey for reference
    sns.violinplot(y=non_outliers_df[feature], color='lightgray', linewidth=1.0, ax=ax, alpha=0.3)
    ax.set_title(title, fontweight='bold', fontsize=12)

plt.suptitle('Cluster Profiles — RFM Feature Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ─── Cluster centroid summary table ──────────────────────────────────────────
# This directly answers Business Questions Q5 and Q6
# Shows mean and median per cluster for interpretability
cluster_summary = (
    non_outliers_df.groupby('Cluster')[['MonetaryValue', 'Frequency', 'Recency']]
    .agg(['mean', 'median', 'count'])
    .round(1)
)
print("Cluster Centroid Summary Table:")
print(cluster_summary.to_string())


In [ ]:
# ─── Assign human-readable business labels to each cluster ──────────────────
# Labels are action verbs — directly usable by marketing teams as campaign targets
cluster_labels_map = {
    0: "RETAIN",      # Mid-range spend and frequency, moderate recency — keep engaged
    1: "RE-ENGAGE",   # High recency (lapsed), low frequency — churn risk, need win-back
    2: "NURTURE",     # Low spend, low frequency — early-stage or irregular customers
    3: "REWARD"       # Low recency, decent spend/frequency — active loyalists
}

non_outliers_df['ClusterLabel'] = non_outliers_df['Cluster'].map(cluster_labels_map)
print("Label distribution (core K-Means clusters):")
print(non_outliers_df['ClusterLabel'].value_counts().to_string())


---
<a id='13'></a>
## 13. Outlier Customer Segments

Customers excluded from K-Means (extreme Monetary and/or Frequency values) are now handled as **premium segments**. These are the most strategically important customers for the business.

| Cluster ID | Definition | Label | Strategic Priority |
|-----------|-----------|-------|-------------------|
| -1 | Monetary outlier only | **PAMPER** | High value per purchase; offer exclusive perks |
| -2 | Frequency outlier only | **UPSELL** | Buy very often but lower spend; grow basket size |
| -3 | Both Monetary AND Frequency outlier | **DELIGHT** | True Champions; white-glove service |


In [ ]:
# ─── Separate outlier customers into 3 distinct premium groups ───────────────
# overlap_indices = customers who are outliers in BOTH Monetary AND Frequency
overlap_indices = monetary_outliers_df.index.intersection(frequency_outliers_df.index)

monetary_only_outliers             = monetary_outliers_df.drop(overlap_indices).copy()
frequency_only_outliers            = frequency_outliers_df.drop(overlap_indices).copy()
monetary_and_frequency_outliers    = monetary_outliers_df.loc[overlap_indices].copy()

# Assign negative cluster IDs to distinguish from the core K-Means clusters
monetary_only_outliers["Cluster"]           = -1   # PAMPER
frequency_only_outliers["Cluster"]          = -2   # UPSELL
monetary_and_frequency_outliers["Cluster"]  = -3   # DELIGHT

outlier_clusters_df = pd.concat([
    monetary_only_outliers,
    frequency_only_outliers,
    monetary_and_frequency_outliers
])

print("Premium segment sizes:")
print(f"  PAMPER  (-1, high monetary only) : {len(monetary_only_outliers):>4} customers")
print(f"  UPSELL  (-2, high freq only)     : {len(frequency_only_outliers):>4} customers")
print(f"  DELIGHT (-3, both outliers)      : {len(monetary_and_frequency_outliers):>4} customers")


In [ ]:
# ─── Violin plots for premium segment profiles ────────────────────────────────
cluster_colors_outliers = {-1: '#9467bd', -2: '#8c564b', -3: '#e377c2'}
fig, axes = plt.subplots(3, 1, figsize=(12, 14))

for ax, feature, title in zip(
    axes,
    ['MonetaryValue', 'Frequency', 'Recency'],
    ['💰 Monetary — Premium Segments', '🔁 Frequency — Premium Segments', '📅 Recency — Premium Segments']
):
    sns.violinplot(x=outlier_clusters_df['Cluster'], y=outlier_clusters_df[feature],
                   palette=cluster_colors_outliers, hue=outlier_clusters_df['Cluster'],
                   ax=ax, inner='quartile', legend=False)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xticklabels(['PAMPER (-1)', 'UPSELL (-2)', 'DELIGHT (-3)'])

plt.suptitle('Premium Customer Segments — Profile Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
<a id='14'></a>
## 14. Final Dashboard — All Segments

Combining all 7 segments (4 core K-Means + 3 outlier premium) into a single unified view for business stakeholders.


In [ ]:
# ─── Merge all clusters into one master DataFrame ────────────────────────────
# Add human-readable labels to outlier segments
outlier_label_map = {-1: "PAMPER", -2: "UPSELL", -3: "DELIGHT"}
outlier_clusters_df['ClusterLabel'] = outlier_clusters_df['Cluster'].map(outlier_label_map)

# Concatenate core K-Means segments + premium outlier segments
full_clustering_df = pd.concat([non_outliers_df, outlier_clusters_df])

# Drop the LastInvoiceDate column — not needed for reporting
full_clustering_df = full_clustering_df.drop(columns=['LastInvoiceDate'], errors='ignore')

print(f"Total customers classified: {len(full_clustering_df):,}")
print("\nFull segment distribution:")
print(full_clustering_df['ClusterLabel'].value_counts().to_string())


In [ ]:
# ─── Dual-axis dashboard: segment sizes + mean RFM lines ─────────────────────
# Bars = number of customers in each segment
# Lines = average Recency, Frequency, Monetary/100 per segment
# Gives marketing leads a single-view summary of all 7 segments

cluster_counts = full_clustering_df['ClusterLabel'].value_counts()
full_clustering_df["MonetaryValue per 100 pounds"] = full_clustering_df["MonetaryValue"] / 100.0
feature_means = full_clustering_df.groupby('ClusterLabel')[
    ['Recency', 'Frequency', 'MonetaryValue per 100 pounds']
].mean()

fig, ax1 = plt.subplots(figsize=(14, 7))

# Bar chart — customer count per segment
sns.barplot(x=cluster_counts.index, y=cluster_counts.values, ax=ax1,
            palette='viridis', hue=cluster_counts.index, legend=False)
ax1.set_ylabel('Number of Customers', color='navy', fontsize=12)
ax1.set_xlabel('Customer Segment', fontsize=12)
ax1.set_title('📊 All Customer Segments: Size + Average RFM Features', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=15)

# Line plot — mean feature values on right-hand axis
ax2 = ax1.twinx()
sns.lineplot(data=feature_means, ax=ax2, palette='Set2', marker='o', linewidth=2.2)
ax2.set_ylabel('Average Feature Value', color='green', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# ─── Heatmap: normalised RFM profile per segment ─────────────────────────────
# Each cell shows actual mean value (annotated) on a relative colour scale
# Green = low value, Red = high value (important to interpret by feature direction)
# For RECENCY: red = more lapsed (bad); for MONETARY: red = richer (good)

segment_heatmap = full_clustering_df.groupby('ClusterLabel')[
    ['Recency', 'Frequency', 'MonetaryValue']
].mean()

# Normalise to 0-1 range so all 3 metrics display on the same colour scale
segment_heatmap_norm = (
    (segment_heatmap - segment_heatmap.min()) /
    (segment_heatmap.max() - segment_heatmap.min())
)

plt.figure(figsize=(10, 5))
sns.heatmap(segment_heatmap_norm,
            annot=segment_heatmap.round(1),
            fmt='.1f',
            cmap='RdYlGn_r',
            linewidths=0.5,
            cbar_kws={'label': 'Normalised Score (0=low, 1=high)'})
plt.title('🗺️ Segment RFM Heatmap (Normalised)', fontsize=14, fontweight='bold')
plt.xlabel('RFM Feature')
plt.ylabel('Customer Segment')
plt.tight_layout()
plt.show()


---
<a id='15'></a>
## 15. Business Recommendations

Based on the RFM segment profiles, here are concrete, actionable strategies for each group:

---

### 🏆 DELIGHT (Cluster -3) — True Champions
> *High Monetary + High Frequency + Recently Active*

**Who they are:** The absolute top-tier — spend the most, buy the most often, still active.

**Recommended Actions:**
- Invite to an exclusive VIP loyalty programme with early product access
- Assign a dedicated account manager for B2B wholesalers in this group
- Send personalised thank-you gifts or curated bundles
- Leverage as brand ambassadors — ask for reviews and referrals

---

### 💎 PAMPER (Cluster -1) — High Value Buyers
> *High Monetary spend per visit but not necessarily high frequency*

**Who they are:** High-ticket purchasers — big order values, less frequent visits.

**Recommended Actions:**
- Focus on Average Order Value (AOV): offer premium bundles
- Seasonal outreach around Christmas, Valentine's, Mother's Day (peak gift periods)
- Volume pricing tiers to encourage repeat high-value orders

---

### 🔄 UPSELL (Cluster -2) — Frequent Bargain Hunters
> *High Frequency but lower spend per visit*

**Who they are:** Very engaged customers who buy often but spend less per basket.

**Recommended Actions:**
- Cross-sell higher-margin categories they haven't purchased yet
- "Buy 5, get 15% off" type volume discount thresholds to grow basket size
- Introduce to premium product lines via targeted email

---

### 💛 REWARD (Cluster 3) — Active Loyalists
> *Low Recency, decent Frequency and Monetary spend*

**Who they are:** Consistent, engaged customers who are performing well.

**Recommended Actions:**
- Enrol in a points-based rewards scheme
- Offer referral bonuses — leverage word-of-mouth
- Periodic exclusive flash sales to reinforce loyalty

---

### 🟢 RETAIN (Cluster 0) — Stable Mid-Tier
> *Moderate across all dimensions*

**Who they are:** The steady backbone — not biggest spenders but reliable.

**Recommended Actions:**
- Regular email newsletter with product highlights and new arrivals
- Seasonal promotions to encourage incremental purchases
- "We miss you" nudge if Recency starts increasing

---

### 🌱 NURTURE (Cluster 2) — New / Occasional Buyers
> *Low spend, low frequency*

**Who they are:** Recently acquired or irregular — highest potential for growth.

**Recommended Actions:**
- Onboarding email sequence with product education
- "First repeat purchase" discount to drive habit formation
- Collect feedback on their initial purchase experience

---

### 🔴 RE-ENGAGE (Cluster 1) — At-Risk / Lapsed
> *High Recency (haven't purchased in a long time)*

**Who they are:** Once-active customers who have gone quiet — churn risk.

**Recommended Actions:**
- Win-back campaign: "We haven't seen you in a while — here's 20% off"
- Survey to understand the reason for disengagement
- Last-chance urgency message before marking as churned
- After 12+ months of inactivity, remove from active lists to protect email deliverability

---

### 💰 Suggested Marketing Budget Allocation

| Segment | Priority | Budget Share |
|---------|----------|-------------|
| DELIGHT | 🔴 Critical | 25% |
| PAMPER | 🔴 High | 20% |
| REWARD | 🟡 High | 15% |
| UPSELL | 🟡 Medium | 15% |
| RETAIN | 🟢 Medium | 10% |
| NURTURE | 🟢 Low | 10% |
| RE-ENGAGE | ⚫ Low | 5% |


---
<a id='16'></a>
## 16. Conclusion & Next Steps

### ✅ Project Summary

| Item | Result |
|------|--------|
| Raw transactions | 525,461 |
| After data cleaning | ~406,000 |
| Unique customers segmented | ~4,300 |
| Core K-Means clusters | 4 |
| Premium outlier segments | 3 |
| **Total named segments** | **7** |
| Optimal k (elbow + silhouette) | 4 |

The project successfully identified **7 meaningful customer archetypes** from transactional data using RFM analysis and K-Means clustering. Each segment maps directly to a business-actionable CRM strategy.

---

### 🚀 Ideas to Make This Project Even More Unique

#### 🔬 Analytical Enhancements
| Idea | Benefit |
|------|---------|
| **RFM Scoring (1–5 per dimension)** | Composite RFM score adds richer profiling before clustering |
| **PCA Visualisation** | Reduce 3D RFM to 2D for cleaner interpretable cluster plots |
| **DBSCAN or Hierarchical Clustering** | Compare — do natural density-based clusters differ from K-Means? |
| **Gaussian Mixture Models (GMM)** | Soft clustering: gives probability of belonging to each segment |
| **Log-transform Monetary** | Reduces right-skew before scaling — may improve cluster separation |

#### 📈 Business Intelligence Additions
| Idea | Benefit |
|------|---------|
| **Country-level breakdown** | Which countries have the most VIPs? Any geographic patterns? |
| **Seasonal trend analysis** | Do segment sizes shift over time? (pre-Christmas vs January slump) |
| **Product affinity per segment** | What do DELIGHT customers buy that NURTURE customers don't? |
| **Churn prediction classifier** | Train a model to predict next month's segment transition |
| **Customer Lifetime Value (CLV)** | Project future revenue using segment-level purchase rates |

#### 🛠️ Engineering & Deployment
| Idea | Benefit |
|------|---------|
| **Monthly re-scoring pipeline** | Automate RFM recomputation and segment assignment each month |
| **Export to CRM / Salesforce** | Push segment labels directly into marketing automation tools |
| **Streamlit / Flask dashboard** | Interactive web app for non-technical marketing stakeholders |
| **A/B test tracking per segment** | Measure whether targeted campaigns outperform generic ones |

---

### 📚 References
- Buttle, F. (2009). *Customer Relationship Management*. Routledge.
- Chen, D. et al. (2012). UCI Online Retail II Dataset. [UCI ML Repository](https://archive.ics.uci.edu/dataset/502/online+retail+ii)
- Scikit-Learn Documentation: [KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- RFM Analysis: [Wikipedia](https://en.wikipedia.org/wiki/RFM_(market_research))

---
*Capstone project demonstrating end-to-end unsupervised machine learning on real-world e-commerce transactional data.*
